# 3.3 SES Prototype — Space Weather (Reproducible GitHub Version)

**Purpose**  
Generate Space Weather artefacts required by the Streamlit dashboard, while remaining fully reproducible from the GitHub repository.

**Required outputs**  
- `data/processed/ses_spaceweather_dataset.csv`  
- `reports/figures/spaceweather_risky_shap_values.csv`  
- `reports/figures/spaceweather_continuous_heatmap.png`

**XAI note**  
SHAP explanations are generated using a surrogate Random Forest model trained to approximate a Space Weather risk proxy.


In [ ]:
from __future__ import annotations
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
import matplotlib.pyplot as plt
import shap


In [ ]:
# Resolve repo root
HERE = Path.cwd().resolve()
REPO = HERE
while not (REPO / "app.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
assert (REPO / "app.py").exists(), "Repo root not found"

DATA_P = REPO / "data" / "processed"
DATA_I = REPO / "data" / "interim"
FIGS   = REPO / "reports" / "figures"

DATA_P.mkdir(parents=True, exist_ok=True)
FIGS.mkdir(parents=True, exist_ok=True)

IN_SW = DATA_P / "ses_spaceweather_dataset.csv"
if not IN_SW.exists() and (DATA_I / "ses_spaceweather_dataset.csv").exists():
    IN_SW = DATA_I / "ses_spaceweather_dataset.csv"

OUT_DATA = DATA_P / "ses_spaceweather_dataset.csv"
OUT_SHAP = FIGS / "spaceweather_risky_shap_values.csv"
OUT_PNG  = FIGS / "spaceweather_continuous_heatmap.png"

print("Repo:", REPO)


In [ ]:
# Load or synthesise data
def synth(n=500):
    rng = np.random.default_rng(42)
    t = pd.date_range("2021-11-01", periods=n, freq="2H", tz="UTC")
    kp = np.clip(rng.normal(2.5, 1.0, n), 0, 9)
    att = np.abs(rng.normal(0.02, 0.01, n))
    temp = rng.normal(55, 6, n)
    man = rng.choice([0,1], n, p=[0.85,0.15])

    df = pd.DataFrame({
        "time": t,
        "kp": kp,
        "thruster_temp_c": temp,
        "attitude_error_deg": att,
        "maneuver_flag": man,
        "maneuver_type": np.where(man==1,"station_keeping","none")
    })

    kp_n = (kp-kp.min())/(kp.max()-kp.min()+1e-12)
    att_n = (att-att.min())/(att.max()-att.min()+1e-12)
    temp_n = (temp-temp.min())/(temp.max()-temp.min()+1e-12)

    df["risk_score"] = (0.45*kp_n+0.35*att_n+0.2*temp_n)*(0.6+0.4*man)
    return df

if IN_SW.exists():
    sw = pd.read_csv(IN_SW)
    if "time" not in sw.columns:
        sw = sw.rename(columns={sw.columns[0]:"time"})
    sw["time"] = pd.to_datetime(sw["time"], utc=True, errors="coerce")
    if "risk_score" not in sw.columns:
        sw["risk_score"] = 0.0
else:
    sw = synth()

sw = sw.replace([np.inf,-np.inf],np.nan).fillna(method="ffill").fillna(0.0)
sw.to_csv(OUT_DATA, index=False)
print("Saved dataset:", OUT_DATA)


In [ ]:
# SHAP surrogate
features = [c for c in sw.columns if c not in ("time","maneuver_type") and pd.api.types.is_numeric_dtype(sw[c])]
X = sw[features].values
y = sw["risk_score"].values

X = StandardScaler().fit_transform(X)

rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
rf.fit(X,y)

explainer = shap.TreeExplainer(rf)

top = sw.sort_values("risk_score", ascending=False).head(30)
idx = top.index.to_numpy()

shap_vals = explainer.shap_values(X[idx])
shap_mat = np.asarray(shap_vals).T

labels = [f"m{i}" for i in range(shap_mat.shape[1])]
pd.DataFrame(shap_mat, index=features, columns=labels).to_csv(OUT_SHAP)
print("Saved SHAP:", OUT_SHAP)


In [ ]:
# Continuous heatmap
rng = np.random.default_rng(42)
sel = rng.choice(len(sw), size=min(200,len(sw)), replace=False)
sh = np.asarray(explainer.shap_values(X[sel]))

mean_abs = np.mean(np.abs(sh), axis=0)
top_idx = np.argsort(mean_abs)[::-1][:min(18,len(features))]
cont = sh[:,top_idx].T

plt.figure(figsize=(12,6))
plt.imshow(cont, aspect="auto")
plt.yticks(range(len(top_idx)), [features[i] for i in top_idx], fontsize=7)
plt.title("Space Weather – Continuous SHAP overview (surrogate explanation)")
plt.tight_layout()
plt.savefig(OUT_PNG, dpi=200)
plt.close()

print("Saved heatmap:", OUT_PNG)
